# B2.3 · Vulnerability auditing: three generations of SAST

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.2 · Threat modelling from what the estate already knows](https://spbreed.github.io/cyber-commons/lessons/B2.2.html)**.

| | |
|---|---|
| Tools used | OpenGrep, Semgrep OSS, CodeQL, GLM-4.6, Kimi K2, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Score grep, taint rules and model review against the same corpus, then combine them behind a confidence gate.

**Why a security engineer needs it.** Pattern matching floods the queue; the false-positive rate is what actually changed. The control it builds is: stage 7: deterministic rules for what rules do well, model reasoning for what rules cannot express.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Three generations of static analysis are on the market and all three are sold with the same word. Pattern matching cannot follow a value; dataflow cannot read intent; a model can do both and will also tell you about a vulnerability that is not there.

> **At CyberTravels.** The IDOR that exposed card details by booking ID (R8) is exactly the class each generation of SAST handles differently — and the class the third generation will also confidently invent.

## 2 · The framework

```
   gen 1  pattern      grep-shaped     finds: the literal string
                                       misses: the same bug spelled differently

   gen 2  dataflow     source -> sink  finds: the value that reaches
                                       misses: intent, framework magic

   gen 3  reasoning    reads it        finds: both of the above
                                       adds:  confident findings that are not real

   the third generation does not replace the second. it needs it as an oracle.
```

**Stage 7 — Vulnerability auditing.** The deep-dive analysis stage, and the one
people think of as "SAST". It has had three generations, and knowing what each
can and cannot see is what stops you buying the wrong one.

**Generation 1 — grep.** Pattern-match dangerous constructs. Fast, zero setup,
fires on every occurrence whether reachable or not. Precision is poor, so it gets
muted.

**Generation 2 — rules with dataflow.** Semgrep, CodeQL, OpenGrep. Parse to an
AST or graph and track *taint*: does untrusted input reach a dangerous sink?
Precision improves enormously. The cost is that a rule only finds the pattern
someone wrote it for.

**Generation 3 — model review.** An open-weight model reads the code and reasons.
No rule needs to exist first, which is exactly its value — and it also invents
bugs that are not there, confidently.

The mistake is treating generation 3 as a replacement for generation 2. The
combination that works: rules for what rules do well, deterministically; the
model for what rules cannot express; and everything the model says treated as a
**hypothesis** until stages 8–12 confirm it.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Generation 1 — grep, and why it gets muted

The safe functions in this corpus matter more than the buggy ones: a scanner that fires on parameterised SQL is one nobody runs twice.

## 4 · Generation 2 — taint rules

The improvement is not a better pattern. It is a different question: *does untrusted input reach this sink?* A function parameter is untrusted; a string literal is not.

## 5 · Generation 3 — what rules structurally cannot see

Generation 2 is perfect on this corpus. So why involve a model? Because a rule only finds what someone wrote it for. Here is a bug with no rule: an authorization check that is *present* and wrong.

## 6 · Generation 2, as the tool you would actually run

The taint engine above is forty lines so it fits in a lesson. In production
generation 2 is Semgrep, CodeQL or OpenGrep, and a rule is a file. This is the
Semgrep rule for the same taint property the engine above implements:

```yaml
rules:
  - id: cybertravels-sql-concat
    languages: [python]
    severity: ERROR
    message: >-
      Traveller-controlled input is concatenated into a SQL string. Use a
      parameterised query.
    mode: taint
    pattern-sources:
      - pattern: $REQ.args[...]
      - pattern: $REQ.files[...]
    pattern-sinks:
      - pattern: $CONN.execute(...)
    pattern-sanitizers:
      - pattern: sqlite3.paramstyle
```

[`labs/tools/semgrep-sast/`](https://github.com/spbreed/cyber-commons/tree/main/labs/tools/semgrep-sast)
installs Semgrep 1.176.0 and runs it against a pull request from the Coding
Agent. Two things came out of that run and both matter here.

**Coverage is a configuration decision, and it is invisible.** The same file,
two ruleset widths:

```
  p/python + p/secrets: 1 finding
    line  17  ERROR   subprocess-shell-true

  seven packs: 4 findings
    line   9  ERROR   sqlalchemy-execute-raw-query
    line  14  WARNING eval-detected
    line  17  ERROR   subprocess-shell-true
    line  20  ERROR   disabled-cert-validation
```

Nothing about the file changed. On the narrow setting three real defects were
simply not looked for, and the scan exits 0 either way.

**And two defects survived both widths:**

```
  line  22  MISSED a live-looking API key on a module-level constant
  line   7  MISSED find_booking performs no authorisation check of any kind
```

The first is lexical — `p/secrets` was enabled and did not fire, because the
string matches no known provider's format. A rule could catch it, once someone
writes that rule. The second cannot be caught by any rule, because the defect is
the **absence** of a call in a function whose caller holds payments scope. That
is the boundary generation 3 exists to cross, and it is why the answer is
"both" rather than "the newer one".

## 7 · An agent drives both, because you cannot afford to run both everywhere

Generation 2 is cheap enough to run over the whole repository. Generation 3 is
not — at four million lines the model pass costs more than the finding is
worth, and a model asked to review everything reviews nothing carefully.

So neither generation is the interesting part. **The allocation is.** An agent
sits above both, and its policy is three rules:

1. run the deterministic scanner everywhere, with the widest ruleset that is
   not noisy, because it is nearly free;
2. spend the model pass only where stage 1 said risk lives **and** the rules
   were silent — silence in a high-risk zone is the signal, not the noise;
3. mark everything the model says as a hypothesis, never a finding, because
   stages 8 to 12 are what turn one into the other.

## 8 · The stage, as a skill

Three generations of analysis over the same CyberTravels code, and they fail differently: grep flags the safe queries, taint finds the real flows and nothing in `authz.py`, and the model finds the authorization defect that has no syntactic signature — along with the hallucination that is the price of it. The skill runs all three and reports precision, recall and that last column.

In [ ]:
# skills/appsec/sast-generation-comparison/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: sast-generation-comparison
description: >-
  Run pattern rules, taint analysis and a model over the same code and compare
  precision, recall and the class of defect only the third one finds. Use when
  choosing static analysis, justifying a model in the pipeline, or explaining
  why the scanner's output is mostly noise.
allowed-tools: Read, Grep, Glob
---

# Three generations, and they fail differently

Grep-class rules match syntax and cannot see data flow, so they flag the
parameterised query and the constant insert. Taint analysis follows source to
sink and is precise on the bugs it models. A model reads intent and finds the
class neither of the others can express — an authorization defect, where nothing
is malformed and the code is simply wrong about who may do what.

## When to use this

Choosing or defending a static analysis stack, and any time somebody proposes
replacing one generation with another rather than layering them.

## Procedure

**1 — Assemble a corpus with known ground truth.** Real bugs, safe lookalikes
of each bug, and at least one defect with no syntactic signature. The
lookalikes are what produce the precision number; without them every tool looks
perfect.

**2 — Run generation 1: pattern rules.** Record every hit and mark it against
ground truth. Precision here is usually about half, and the false positives are
the safe versions of the true positives.

**3 — Run generation 2: taint rules.** Source, sink, sanitiser. Expect high
precision and recall inside the model it has, and expect it to find nothing in
the file whose defect is not a flow.

**4 — Run generation 3: a model, with confidence.** Give it the same code. Record
what it finds, its confidence, and — separately — anything it asserts that is
not in the file. That last column is the cost of this generation.

**5 — Report per generation and per defect class.** The useful output is not a
winner; it is which class each generation can and cannot express, and the
precision each pays for its recall.

## Output contract

```json
{
  "corpus": [{"file": "str", "defect": "str|null", "cwe": "str|null"}],
  "generations": [{"name": "grep|taint|model", "findings": 0, "true_positives": 0,
                   "precision": 0.0, "recall": 0.0, "hallucinated": 0}],
  "only_found_by": [{"defect": "str", "generation": "str"}],
  "recommendation": {"layers": ["str"], "why": "str"}
}
```

## Failure modes

- **A corpus with no safe lookalikes.** Precision becomes meaningless.
- **Comparing on recall alone.** Grep has excellent recall and unusable
  precision.
- **Not counting the model's hallucinations.** They are the reason generation 3
  needs generation 4 — verification.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, os, shutil, sys

# Make the shared runtime importable, then import it. On Kaggle an attached
# kernel is mounted as __script__.py — not on sys.path and not named after the
# kernel — so copy it to the name it is imported by. Locally it is already a
# file of that name in the repository.
_k = glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py", recursive=True)
if _k:
    shutil.copy(_k[0], "cyber_commons_skill_runtime.py")
sys.path[:0] = [".", "skills/_runtime", "../skills/_runtime", "../../skills/_runtime"]

from cyber_commons_skill_runtime import run_skill

# Split skills/appsec/sast-generation-comparison/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/appsec/sast-generation-comparison/scripts/sast_generation_comparison.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Run grep rules, taint rules and a model over the same code and compare precision, recall and what only the third one finds.

This is the executable half of the `sast-generation-comparison` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

# --- model backend: replay by default, a Kaggle open-weight model when served -
# One URL and one header shape, no vendor SDK. Standard library only, so the
# notebook stays self-contained.
# The model adapter comes from the shared runtime, not from a copy in this
# file. In a lesson notebook the cell above has already loaded it; standalone,
# find it the same way that cell does.
# The runtime comes from the shared library. The lesson cell above put it
# on the path; standalone, PYTHONPATH does (see scripts/test_skills.py).
from cyber_commons_skill_runtime import announce_backend, ask

announce_backend()


CODE = {
"db.py": '''
def get_user(conn, name):
    # BUG: user input concatenated into SQL
    return conn.execute("SELECT * FROM users WHERE name = \'" + name + "\'")

def get_user_safe(conn, name):
    # parameterised — the driver escapes it
    return conn.execute("SELECT * FROM users WHERE name = ?", (name,))

def audit_note(conn, msg):
    # a constant string. No user input anywhere.
    return conn.execute("INSERT INTO audit(msg) VALUES (\'startup\')")
''',
"ops.py": '''
import os, subprocess

def ping(host):
    # BUG: shell string built from user input
    os.system("ping -c1 " + host)

def ping_safe(host):
    subprocess.run(["ping", "-c1", host], check=True)
''',
"files.py": '''
def read_doc(base, filename):
    # BUG: path joined from untrusted input
    return open(base + "/" + filename).read()
''',
}
import re
GREP_RULES = [("CWE-89","SQL injection",r"execute\("),
              ("CWE-78","command injection",r"os\.system|subprocess"),
              ("CWE-22","path traversal",r"open\(")]
def gen1(code):
    return [(cwe, name, f, i, ln.strip())
            for f, src in code.items()
            for i, ln in enumerate(src.splitlines(), 1)
            for cwe, name, pat in GREP_RULES if re.search(pat, ln)]

g1 = gen1(CODE)
print(f"generation 1 (grep): {len(g1)} findings")
for cwe, name, f, i, ln in g1:
    print(f"   {cwe:8s}{f}:{i:<3} {ln[:52]}")

TRUTH = {("CWE-89","db.py",4), ("CWE-78","ops.py",6), ("CWE-22","files.py",4)}
def score(findings, label):
    got = {(c, f, i) for c, _, f, i, _ in findings}
    tp, fp, fn = len(got & TRUTH), len(got - TRUTH), len(TRUTH - got)
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec  = tp/(tp+fn) if tp+fn else 0.0
    print(f"{label:32s} tp={tp} fp={fp} fn={fn}  precision={prec:.2f} recall={rec:.2f}")
    return prec, rec
score(g1, "generation 1 · grep")
print("\nfalse positives:")
for cwe, name, f, i, ln in g1:
    if (cwe, f, i) not in TRUTH: print(f"   {f}:{i:<3} {ln[:56]}")

import ast
class TaintRule:
    SINKS = {"execute": ("CWE-89","SQL injection"),
             "system":  ("CWE-78","command injection"),
             "open":    ("CWE-22","path traversal")}
    def scan(self, fname, src):
        out = []
        for fn in [n for n in ast.walk(ast.parse(src)) if isinstance(n, ast.FunctionDef)]:
            tainted = {a.arg for a in fn.args.args}
            for call in [n for n in ast.walk(fn) if isinstance(n, ast.Call)]:
                sink = (call.func.attr if isinstance(call.func, ast.Attribute)
                        else getattr(call.func, "id", ""))
                if sink not in self.SINKS: continue
                cwe, name = self.SINKS[sink]
                for arg in call.args:
                    if self._concat_taint(arg, tainted):
                        out.append((cwe, name, fname, call.lineno,
                                    ast.get_source_segment(src, call) or ""))
                        break
        return out
    @staticmethod
    def _concat_taint(node, tainted):
        for n in ast.walk(node):
            if isinstance(n, ast.BinOp) and isinstance(n.op, ast.Add):
                if {x.id for x in ast.walk(n) if isinstance(x, ast.Name)} & tainted:
                    return True
        return False

rule = TaintRule()
g2 = [f for n, s in CODE.items() for f in rule.scan(n, s)]
print(f"generation 2 (taint rules): {len(g2)} findings")
for cwe, name, f, i, snip in g2: print(f"   {cwe:8s}{f}:{i:<3} {snip[:52]}")
print()
score(g2, "generation 2 · taint rules")

CODE["authz.py"] = '''
def can_delete(user, doc):
    # Reads "or" where it means "and". No sink, no taint, no pattern.
    if user.is_admin or user.id == doc.owner_id or doc.is_public:
        return True
    return False
'''
print("generation 1 on authz.py:", gen1({"authz.py": CODE["authz.py"]}) or "nothing")
print("generation 2 on authz.py:", rule.scan("authz.py", CODE["authz.py"]) or "nothing")

class StandIn:
    """DETERMINISTIC STAND-IN — not a language model. See the note above."""
    KNOWN = {
     "authz.py": [{"cwe":"CWE-863","line":4,"confidence":0.82,
                   "rationale":"disjunctive permission check: a non-public document "
                               "owned by another user is deletable whenever is_public "
                               "is true, and delete rights are never checked"}],
     "db.py": [{"cwe":"CWE-89","line":4,"confidence":0.95,
                "rationale":"name is concatenated into the query string"},
               {"cwe":"CWE-89","line":12,"confidence":0.41,
                "rationale":"audit_note also calls execute"}],     # HALLUCINATION
    }
    def review(self, fname, src): return self.KNOWN.get(fname, [])

model = StandIn()
print("\ngeneration 3 (model review):")
for fname in ("authz.py", "db.py"):
    for f in model.review(fname, CODE[fname]):
        print(f"   {f['cwe']:9s}{fname}:{f['line']:<3} conf={f['confidence']:.2f}  "
              f"{f['rationale'][:52]}")
print("\nIt found the authorization bug neither earlier generation can see.")
print("It also invented a SQL injection in a function with a constant string.")

# What stage 1 said, and what generation 2 found. The agent has both.
HISTORICAL_RISK = {"db.py": 0.53, "authz.py": 0.48, "ops.py": 0.19,
                   "safe.py": 0.02}
MODEL_COST_PER_FILE = 0.031      # dollars, measured on a small open-weight model

def audit_agent(code, rule, model, risk, gate=0.70, risk_floor=0.30):
    """Stage 7, allocated. Rules everywhere; the model where rules went quiet."""
    findings, suppressed, plan = [], [], []
    rule_hits = {f: rule.scan(f, s) for f, s in sorted(code.items())}

    for fname, hits in rule_hits.items():
        for cwe, name, f, i, snip in hits:
            findings.append({"src": "rules", "cwe": cwe, "file": f, "line": i,
                             "confidence": 1.0, "status": "confirmed-by-rule"})

    for fname in sorted(code):
        r = risk.get(fname, 0.0)
        quiet = not rule_hits[fname]
        if r >= risk_floor and quiet:
            plan.append((fname, r, "high risk, rules silent -> REVIEW"))
        elif r >= risk_floor:
            plan.append((fname, r, "high risk, rules already fired -> skip"))
        else:
            plan.append((fname, r, "low historical risk -> skip"))

    reviewed = [f for f, _, why in plan if why.endswith("REVIEW")]
    for fname in reviewed:
        for m in model.review(fname, code[fname]):
            row = {"src": "model", "cwe": m["cwe"], "file": fname,
                   "line": m["line"], "confidence": m["confidence"],
                   "status": "HYPOTHESIS"}
            (findings if m["confidence"] >= gate else suppressed).append(row)

    seen, dedup = set(), []
    for f in sorted(findings, key=lambda r: (r["src"], r["file"], r["line"])):
        k = (f["cwe"], f["file"], f["line"])
        if k not in seen:
            seen.add(k); dedup.append(f)
    return dedup, suppressed, plan, reviewed

final, suppressed, plan, reviewed = audit_agent(CODE, rule, model, HISTORICAL_RISK)

print("the agent's allocation:")
for fname, r, why in plan:
    print(f"   {fname:12s}risk {r:.2f}   {why}")
print(f"\nmodel pass on {len(reviewed)} of {len(CODE)} files "
      f"(${len(reviewed) * MODEL_COST_PER_FILE:.3f} rather than "
      f"${len(CODE) * MODEL_COST_PER_FILE:.3f})")

print(f"\nstage 7 emits {len(final)}, {len(suppressed)} suppressed below 0.70")
for f in final:
    print(f"   [{f['src']:5s}] {f['cwe']:9s}{f['file']}:{f['line']:<3} "
          f"conf={f['confidence']:.2f}  {f['status']}")

TRUTH_FULL = TRUTH | {("CWE-863", "authz.py", 4)}
got = {(f["cwe"], f["file"], f["line"]) for f in final}
print(f"\ntp={len(got & TRUTH_FULL)} fp={len(got - TRUTH_FULL)} "
      f"fn={len(TRUTH_FULL - got)}")
assert not (got - TRUTH_FULL) and not (TRUTH_FULL - got)

# The allocation is a bet, so measure what it costs when it loses. Move the
# authorisation bug into a file with LOW historical risk and re-run.
print("the same corpus, with authz.py carrying no history:")
_, _, plan_b, reviewed_b = audit_agent(CODE, rule, model,
                                       {**HISTORICAL_RISK, "authz.py": 0.04})
final_b, _, _, _ = audit_agent(CODE, rule, model,
                               {**HISTORICAL_RISK, "authz.py": 0.04})
got_b = {(f["cwe"], f["file"], f["line"]) for f in final_b}
missed = TRUTH_FULL - got_b
print(f"   model pass on {len(reviewed_b)} file(s): {reviewed_b}")
print(f"   MISSED: {sorted(missed)}")
print()
print("A new file with no history is invisible to the allocator, and the")
print("allocator is what makes generation 3 affordable. The mitigation is not")
print("subtle - review everything a pull request touched regardless of history,")
print("and let the risk floor decide only where to spend the SECOND pass.")
assert missed == {("CWE-863", "authz.py", 4)}
print()
print("Every model finding above is marked HYPOTHESIS. Stages 8 to 12 decide.")

# ------------------------------------ the same task, against a real model
# Offline this is a labelled replay; with an open-weight model served
# from Kaggle it is the same code calling a real one.

TASK = 'Is this function vulnerable? Name the CWE if so, and say which value reaches the sink.\n\ndef report(request):\n    q = "SELECT * FROM orders WHERE ref = \'" + request.args[\'ref\'] + "\'"\n    return db.execute(q)'

REPLAY = "Yes - CWE-89, SQL injection. request.args['ref'] is concatenated directly into the query string and reaches db.execute unsanitised."

answer, used, model = ask(TASK, replay=REPLAY,
            system='You are a code reviewer. Answer in at most three lines.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("names the CWE identifier", "CWE-89" in answer.upper() or "SQL INJECT" in answer.upper())
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, two possible backends. Offline the answer is")
print("the replay and is labelled as one; with a served model it is the model's.")

## What you just proved

Grep produces 6 findings at 50% precision, flagging the parameterised query, the constant insert and the safe subprocess call. Taint rules find exactly the 3 real injection bugs at 100% precision and recall and find nothing in `authz.py`. The model finds the authorization bug at 0.82 confidence and hallucinates one SQL injection at 0.41. The audit agent then runs the rules everywhere and spends the model pass on one file of four — the one where history says risk lives and the rules were silent — emitting 4 findings with zero false positives, every model finding marked as a hypothesis. The last cell shows what the allocation costs when it loses: give `authz.py` no history and the authorization bug is never reviewed.

## Your turn

Two things, and the second is the one people skip. Point the stand-in at a real GLM-4.6 or Kimi K2 through Ollama and run it on `authz.py` ten times — the variance in what it reports, and in its confidence, decides whether you can gate on confidence at all. Then run Semgrep against one of your own repositories at your current ruleset and at seven packs, and count the difference. Whatever that number is, it has been the number all year.

---

**Next → [B2.4 · Deduplication and contextual verification](https://spbreed.github.io/cyber-commons/lessons/B2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*